In [2]:
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"]="-1"
print(torch.cuda.is_available())

False


/home/crisis1/anaconda3/lib/python3.11/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11070). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


In [3]:
!pip install transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, RandomSampler, TensorDataset
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import numpy as np


[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: pip install --upgrade pip


In [4]:
# Set the model name for MuRIL, XLM-BERT, or mBERT
model_name = 'xlm-roberta-base'  # Can replace with 'bert-base-multilingual-cased'

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)

# Load datasets
df_train = pd.read_csv("DataSet/disaster_response_messages_training.csv")
df_val = pd.read_csv("DataSet/disaster_response_messages_validation.csv")
df_test = pd.read_csv("DataSet/disaster_response_messages_test.csv")

/home/crisis1/anaconda3/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_2227351/1688123403.py:9: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv("DataSet/disaster_response_messages_training.csv")


In [5]:
df_train['original'].fillna(df_train['message'], inplace=True)
df_test['original'].fillna(df_test['message'], inplace=True)
df_val['original'].fillna(df_val['message'], inplace=True)

# List of columns to keep
columns_to_keep = ['original','aid_related','weather_related', 'food', 'shelter', 'water']

# Filter columns for df_train, df_test, and df_val
df_train = df_train[columns_to_keep]
df_test = df_test[columns_to_keep]
df_val = df_val[columns_to_keep]

# Drop rows with missing values
df_train.dropna(inplace=True)
df_val.dropna(inplace=True)
df_test.dropna(inplace=True)

In [6]:
# Extract labels
train_labels = df_train.iloc[:, 1:].values.astype(int)
val_labels = df_val.iloc[:, 1:].values.astype(int)
test_labels = df_test.iloc[:, 1:].values.astype(int)

# Rename 'original' column to 'messages' for consistency
df_train.rename(columns={'original': 'messages'}, inplace=True)
df_val.rename(columns={'original': 'messages'}, inplace=True)
df_test.rename(columns={'original': 'messages'}, inplace=True)


In [7]:
# Preprocessing: Tokenization with attention masks
max_len = 128
def tokenize_text(text, max_len):
    return tokenizer.encode_plus(text, max_length=max_len, truncation=True, padding='max_length', return_tensors='pt')


In [8]:
df_train['input_ids'] = df_train['messages'].apply(lambda x: tokenize_text(x, max_len)['input_ids'].squeeze())
df_train['attention_mask'] = df_train['messages'].apply(lambda x: tokenize_text(x, max_len)['attention_mask'].squeeze())

df_val['input_ids'] = df_val['messages'].apply(lambda x: tokenize_text(x, max_len)['input_ids'].squeeze())
df_val['attention_mask'] = df_val['messages'].apply(lambda x: tokenize_text(x, max_len)['attention_mask'].squeeze())

df_test['input_ids'] = df_test['messages'].apply(lambda x: tokenize_text(x, max_len)['input_ids'].squeeze())
df_test['attention_mask'] = df_test['messages'].apply(lambda x: tokenize_text(x, max_len)['attention_mask'].squeeze())


In [9]:
# Convert to Tensors
train_inputs = torch.stack(df_train['input_ids'].values.tolist())
train_attention_masks = torch.stack(df_train['attention_mask'].values.tolist())
train_labels = torch.tensor(train_labels, dtype=torch.float32)

val_inputs = torch.stack(df_val['input_ids'].values.tolist())
val_attention_masks = torch.stack(df_val['attention_mask'].values.tolist())
val_labels = torch.tensor(val_labels, dtype=torch.float32)

test_inputs = torch.stack(df_test['input_ids'].values.tolist())
test_attention_masks = torch.stack(df_test['attention_mask'].values.tolist())
test_labels = torch.tensor(test_labels, dtype=torch.float32)


In [10]:
# Create TensorDatasets
train_dataset = TensorDataset(train_inputs, train_attention_masks, train_labels)
val_dataset = TensorDataset(val_inputs, val_attention_masks, val_labels)
test_dataset = TensorDataset(test_inputs, test_attention_masks, test_labels)

# DataLoader
batch_size = 32
train_dataloader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset), batch_size=batch_size)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

In [11]:
# Device setup
device = torch.device("cpu")
from transformers import XLMRobertaModel

In [12]:

# # Define the policy network using XLM-Roberta
# class PolicyNetwork(nn.Module):
#     def __init__(self):
#         super(PolicyNetwork, self).__init__()
#         self.xlm_roberta = XLMRobertaModel.from_pretrained('xlm-roberta-base')
#         self.fc = nn.Linear(self.xlm_roberta.config.hidden_size, 5)  # Adjust according to the number of labels

#     def forward(self, input_ids, attention_mask):
#         outputs = self.xlm_roberta(input_ids=input_ids, attention_mask=attention_mask)
#         sequence_output = outputs.last_hidden_state[:, 0, :]  # Take the [CLS] token representation
#         logits = self.fc(sequence_output)
#         # print(f"{logits.shape}")
#         return logits, sequence_output

# # Initialize the policy network
# policy_net = PolicyNetwork().to(device)

In [86]:
class PolicyNetwork(nn.Module):
    def __init__(self, mixup_alpha=0.5):
        super(PolicyNetwork, self).__init__()
        self.xlm_roberta = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        
        self.fc1 = nn.Linear(self.xlm_roberta.config.hidden_size, 768)
        self.fc2 = nn.Linear(768, 5)
        
        # Initialize lambda_val as a trainable parameter
        self.lambda_val = nn.Parameter(torch.tensor(0.5))  # Starting value; can be tuned
        self.mixup_alpha = mixup_alpha  # Initial alpha parameter for potential tuning

    def manifold_mixup(self, features_1, features_2):
        lambda_val = torch.clamp(self.lambda_val, min=0.0, max=1.0)
        mixed_features = lambda_val * features_1 + (1 - lambda_val) * features_2
        return mixed_features

    def forward(self, input_ids, attention_mask):
        outputs = self.xlm_roberta(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state[:, 0, :]  # CLS token representation

        # Get features from fc1 without activation for mixup
        transformed_output1_raw = self.fc1(sequence_output)
        
        # Pass through ReLU before feeding to fc2
        transformed_output1_activated = F.relu(transformed_output1_raw)
        # Perform mixup between raw fc1 output and sequence output
        mixed_features = self.manifold_mixup(transformed_output1_raw, sequence_output)
        # output = self.fc2(transformed_output1_activated)
        output = self.fc2( mixed_features)
        
        
        
        return output, mixed_features

policy_net = PolicyNetwork().to(device)

In [87]:
def reinforce_loss(log_probs, rewards, baseline):
    
    advantages = rewards - baseline#
    
    # print(f"shape of rewards - {rewards.shape}")
    # print(f"shape of log_probs - {log_probs.shape}")
    # print(f"advantages - {advantages}")
    min_advantages = rewards / 2
    advantages = torch.maximum(advantages, min_advantages)
    advantages = rewards
    log_probs = log_probs.sum(dim=1)
    # print(f"shape of log_probs after summing - {log_probs.shape}")
    
    # Now, log_probs and advantages have matching shapes
    loss = (log_probs * advantages).mean()
    return loss


In [88]:
# Define reward function
def calculate_reward(predictions, labels):
    rewards = torch.zeros(predictions.size(0), device=predictions.device)
    for i in range(predictions.size(1)):
        tp = (predictions[:, i] == 1) & (labels[:, i] == 1)
        fn = (predictions[:, i] == 0) & (labels[:, i] == 1)
        fp = (predictions[:, i] == 1) & (labels[:, i] == 0)
        tn = (predictions[:, i] == 0) & (labels[:, i] == 0)

        rewards[tp] += 1 
        rewards[fn] -= 3
        rewards[fp] -= 1
        rewards[tn] += 0.5

    return rewards / predictions.size(1)

In [89]:
chunk_len = 32

In [90]:
# # Define training function
# def train(train_loader, policy_net, optimizer, classification_loss_fn, baseline):
#     policy_net.train()
#     EPOCHS = 1
#     alpha = 0.9

#     for epoch in range(EPOCHS):
#         total_loss = 0
#         total_classification_loss = 0
#         total_policy_loss = 0
#         btch = 0
#         for batch in train_loader:
#             btch += 1
#             print(f"Batch = {btch}")
#             input_ids, attention_masks, labels = [x.to(device) for x in batch]
#             sentence_label_achieved = torch.zeros(input_ids.size(0), labels.size(1)).to(device)

#             sentence_log_probs = []
#             classification_losses = []

#             for chunk_start in range(0, input_ids.size(1), chunk_len):
#                 chunk_ids = input_ids[:, chunk_start:chunk_start + chunk_len]
#                 chunk_attention_mask = attention_masks[:, chunk_start:chunk_start + chunk_len]  # Slice the attention mask
#                 logits = policy_net(chunk_ids,chunk_attention_mask)
#                 probs = torch.sigmoid(logits)
#                 sentence_label_achieved = torch.maximum(sentence_label_achieved, (probs > 0.5).float())

#                 log_probs = torch.log(probs + 1e-10)
#                 sentence_log_probs.append(log_probs)

#                 classification_loss = classification_loss_fn(logits, labels)
#                 classification_losses.append(classification_loss)

#             sentence_log_probs = torch.stack(sentence_log_probs, dim=1)  # Shape: [batch_size, num_chunks, num_labels]
#             sentence_log_probs = sentence_log_probs.mean(dim=1)  # Shape: [batch_size, num_labels]

#             rewards = calculate_reward(sentence_label_achieved, labels)

#             # sentence_log_probs = torch.cat(sentence_log_probs, dim=0)
#             policy_loss = reinforce_loss(sentence_log_probs, rewards, baseline)
            
#             baseline = alpha * baseline + (1 - alpha) * rewards.mean().item()
#             baseline = max(baseline , 0)

#             total_classification_loss_batch = torch.stack(classification_losses).sum()
#             print(f"total_classification_loss_batch = {total_classification_loss_batch}")
            
#             total_loss_batch = total_classification_loss_batch + policy_loss
#             print(f"total_loss_batch = {total_loss_batch}")

#             optimizer.zero_grad()
#             total_loss_batch.backward()
#             optimizer.step()

#             total_loss += total_loss_batch.item()
#             total_classification_loss += total_classification_loss_batch.item()
#             total_policy_loss += policy_loss.item()

#         print(f"Epoch {epoch + 1}/{EPOCHS}, Total Loss: {total_loss / len(train_loader)}, "
#               f"Classification Loss: {total_classification_loss / len(train_loader)}, "
#               f"Policy Loss: {total_policy_loss / len(train_loader)}")

# # Run the training
# classification_loss_fn = nn.BCEWithLogitsLoss()
# baseline = 0.0

In [91]:
import torch.nn.functional as F

def contrastive_loss(anchor_embeddings, positive_embeddings, negative_embeddings, margin=1):
    positive_dist = F.pairwise_distance(anchor_embeddings, positive_embeddings)
    negative_dist = F.pairwise_distance(anchor_embeddings, negative_embeddings)
    
    # Contrastive loss pushes positive pairs closer and negative pairs farther apart
    loss = F.relu(positive_dist - negative_dist + margin).mean()
    return loss
# contrastive_loss = contrastive_loss(anchor_embeddings, positive_embeddings, negative_embeddings, margin=0.5).to(device)

In [92]:
def sample_contrastive_pairs(embeddings, labels):
    positive_pairs = []
    negative_pairs = []
    
    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            if torch.dot(labels[i], labels[j]).item() > 0:  # Common label
                positive_pairs.append((embeddings[i], embeddings[j]))
            else:
                negative_pairs.append((embeddings[i], embeddings[j]))

    # Find the minimum number of pairs available
    min_pairs = min(len(positive_pairs), len(negative_pairs))
    
    # Sample equal numbers of positive and negative pairs
    positive_pairs = positive_pairs[:min_pairs]
    negative_pairs = negative_pairs[:min_pairs]
    
    # Stack the sampled pairs into tensors
    positive_embeddings = torch.stack([pair[0] for pair in positive_pairs])
    negative_embeddings = torch.stack([pair[1] for pair in negative_pairs])
    anchor_embeddings = torch.stack([pair[0] for pair in positive_pairs])  # Use positive pairs as anchor

    return anchor_embeddings, positive_embeddings, negative_embeddings


In [93]:
# Updated training function to include contrastive loss
def train(train_loader, policy_net, optimizer, classification_loss_fn, baseline, contrastive_loss_fn, contrastive_margin=0.5):
    policy_net.train()
    EPOCHS = 1
    alpha = 0.9

    for epoch in range(EPOCHS):
        total_loss = 0
        total_classification_loss = 0
        total_policy_loss = 0
        total_contrastive_loss = 0  # Track contrastive loss
        btch = 0

        for batch in train_loader:
            btch += 1
            print(f"batch number = {btch}")
            input_ids, attention_masks, labels = [x.to(device) for x in batch]
            sentence_label_achieved = torch.zeros(input_ids.size(0), labels.size(1)).to(device)
            sentence_log_probs = []
            classification_losses = []
            embeddings = []  # To store embeddings for contrastive learning

            for chunk_start in range(0, input_ids.size(1), chunk_len):
                chunk_ids = input_ids[:, chunk_start:chunk_start + chunk_len]
                chunk_attention_mask = attention_masks[:, chunk_start:chunk_start + chunk_len]
                
                # Forward pass to get logits and embeddings
                logits, sequence_output = policy_net(chunk_ids, chunk_attention_mask)
                embeddings.append(sequence_output)  # Collect embeddings for contrastive loss
                
                # Compute probability and log probability
                probs = torch.sigmoid(logits)
                sentence_label_achieved = torch.maximum(sentence_label_achieved, (probs > 0.5).float())
                log_probs = torch.log(probs + 1e-10)
                sentence_log_probs.append(log_probs)
                
                classification_loss = classification_loss_fn(logits, labels)
                classification_losses.append(classification_loss)

            # Calculate classification and policy loss
            sentence_log_probs = torch.stack(sentence_log_probs, dim=1).mean(dim=1)
            rewards = calculate_reward(sentence_label_achieved, labels)
            policy_loss = reinforce_loss(sentence_log_probs, rewards, baseline)
            
            baseline = alpha * baseline + (1 - alpha) * rewards.mean().item()
            baseline = max(baseline , 0)

            total_classification_loss_batch = torch.stack(classification_losses).sum()
            
            # Contrastive loss
            embeddings = torch.cat(embeddings, dim=0)  # Concatenate embeddings from chunks
            anchor_embeddings, positive_embeddings, negative_embeddings = sample_contrastive_pairs(embeddings, labels)
            # Calculate contrastive loss only if pairs are available
            if anchor_embeddings.size(0) > 0:
                contrastive_loss = contrastive_loss_fn(anchor_embeddings, positive_embeddings, negative_embeddings, 1.0)
            else:
                contrastive_loss = torch.tensor(0.0, device=device)  # No contrastive pairs, set loss to 0
            

            # contrastive_loss = contrastive_loss_fn(embeddings, positive_embeddings, negative_embeddings, contrastive_margin)

            
            print(f"contrastive_loss = {10*contrastive_loss}")
            print(f"total_classification_loss_batch = {total_classification_loss_batch}")
            print(f"policy_loss = {policy_loss}")

            
            # Total loss combines all three objectives
            total_loss_batch = total_classification_loss_batch + policy_loss + 10*contrastive_loss

            print(f"total_loss_batch = {total_loss_batch}")
            
            # Backpropagation and optimization
            optimizer.zero_grad()
            total_loss_batch.backward()
            optimizer.step()

            # Accumulate losses
            total_loss += total_loss_batch.item()
            total_classification_loss += total_classification_loss_batch.item()
            total_policy_loss += policy_loss.item()
            total_contrastive_loss += contrastive_loss.item()

        print(f"Epoch {epoch + 1}/{EPOCHS}, Total Loss: {total_loss / len(train_loader)}, "
              f"Classification Loss: {total_classification_loss / len(train_loader)}, "
              f"Policy Loss: {total_policy_loss / len(train_loader)}, "
              f"Contrastive Loss: {total_contrastive_loss / len(train_loader)}")

# Run the training
classification_loss_fn = nn.BCEWithLogitsLoss()
baseline = 0.0

In [94]:
# Use AdamW optimizer and BCEWithLogitsLoss for multilabel classification
optimizer = torch.optim.AdamW(policy_net.parameters(), lr=2e-5)


In [95]:
train(train_dataloader, policy_net, optimizer, classification_loss_fn, baseline, contrastive_loss_fn=contrastive_loss)

batch number = 1
contrastive_loss = 1.0787229537963867
total_classification_loss_batch = 2.730942964553833
policy_loss = 1.6746408939361572
total_loss_batch = 5.484306812286377
batch number = 2
contrastive_loss = 0.34581679105758667
total_classification_loss_batch = 2.726499557495117
policy_loss = 2.0171520709991455
total_loss_batch = 5.089468002319336
batch number = 3
contrastive_loss = 0.2816901206970215
total_classification_loss_batch = 2.7379159927368164
policy_loss = 2.2007017135620117
total_loss_batch = 5.22030782699585
batch number = 4
contrastive_loss = 0.12435957044363022
total_classification_loss_batch = 2.7297844886779785
policy_loss = 1.5041913986206055
total_loss_batch = 4.358335494995117
batch number = 5
contrastive_loss = 0.3235625624656677
total_classification_loss_batch = 2.707129955291748
policy_loss = 1.0163676738739014
total_loss_batch = 4.047060012817383
batch number = 6
contrastive_loss = 0.20408162474632263
total_classification_loss_batch = 2.6678645610809326
pol

In [47]:
train(train_dataloader, policy_net, optimizer, classification_loss_fn, baseline)

Batch = 1
total_classification_loss_batch = 3.1021742820739746
total_loss_batch = 3.3739819526672363
Batch = 2
total_classification_loss_batch = 2.8785314559936523
total_loss_batch = 3.084266424179077
Batch = 3
total_classification_loss_batch = 2.802734851837158
total_loss_batch = 2.563486099243164
Batch = 4
total_classification_loss_batch = 2.705605983734131
total_loss_batch = 2.2605507373809814
Batch = 5
total_classification_loss_batch = 2.6718056201934814
total_loss_batch = 2.4698657989501953
Batch = 6
total_classification_loss_batch = 2.609072208404541
total_loss_batch = 2.406513214111328
Batch = 7
total_classification_loss_batch = 2.646149158477783
total_loss_batch = 2.1378462314605713
Batch = 8
total_classification_loss_batch = 2.6814327239990234
total_loss_batch = 2.6568472385406494
Batch = 9
total_classification_loss_batch = 2.543976306915283
total_loss_batch = 1.9456298351287842
Batch = 10
total_classification_loss_batch = 2.6814374923706055
total_loss_batch = 3.09677791595459

In [96]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import torch

# Define evaluation function
def evaluate(eval_loader, model):
    model.eval()
    true_labels = []
    predictions = []

    with torch.no_grad():
        b = 0
        for batch in eval_loader:
            b += 1
            print(f"batch = {b}")
            input_ids, attention_masks, labels = [x.to(device) for x in batch]
            logits, sequence_output = policy_net(input_ids,attention_masks)
            probs = torch.sigmoid(logits)

            # Convert probabilities to binary predictions
            pred_labels = (probs >= 0.5).float().cpu().numpy()
            true_labels.extend(labels.cpu().numpy())
            predictions.extend(pred_labels)

    # Convert lists to arrays for easier manipulation
    true_labels = np.array(true_labels)
    predictions = np.array(predictions)
    
    return true_labels, predictions

In [97]:
true_labels, predictions = evaluate(test_dataloader, policy_net)


batch = 1
batch = 2
batch = 3
batch = 4
batch = 5
batch = 6
batch = 7
batch = 8
batch = 9
batch = 10
batch = 11
batch = 12
batch = 13
batch = 14
batch = 15
batch = 16
batch = 17
batch = 18
batch = 19
batch = 20
batch = 21
batch = 22
batch = 23
batch = 24
batch = 25
batch = 26
batch = 27
batch = 28
batch = 29
batch = 30
batch = 31
batch = 32
batch = 33
batch = 34
batch = 35
batch = 36
batch = 37
batch = 38
batch = 39
batch = 40
batch = 41
batch = 42
batch = 43
batch = 44
batch = 45
batch = 46
batch = 47
batch = 48
batch = 49
batch = 50
batch = 51
batch = 52
batch = 53
batch = 54
batch = 55
batch = 56
batch = 57
batch = 58
batch = 59
batch = 60
batch = 61
batch = 62
batch = 63
batch = 64
batch = 65
batch = 66
batch = 67
batch = 68
batch = 69
batch = 70
batch = 71
batch = 72
batch = 73
batch = 74
batch = 75
batch = 76
batch = 77
batch = 78
batch = 79
batch = 80
batch = 81
batch = 82
batch = 83


In [98]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

# Evaluation function with return of true_labels and predictions
def evaluate(dataloader, policy_net, classification_loss_fn):
    policy_net.eval()
    total_loss = 0
    all_labels = []
    all_predictions = []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids, attention_masks, labels = [x.to(device) for x in batch]

            sentence_label_achieved = torch.zeros(input_ids.size(0), labels.size(1)).to(device)
            classification_losses = []
            for chunk_start in range(0, input_ids.size(1), chunk_len):
                chunk_ids = input_ids[:, chunk_start:chunk_start + chunk_len]
                chunk_attention_mask = attention_masks[:, chunk_start:chunk_start + chunk_len]
                logits, sequence_output = policy_net(chunk_ids, chunk_attention_mask)
                probs = torch.sigmoid(logits)
                sentence_label_achieved = torch.maximum(sentence_label_achieved, (probs > 0.5).float())

            # Collect labels and predictions
            all_labels.append(labels.cpu().numpy())
            all_predictions.append(sentence_label_achieved.cpu().numpy())
    
    # Concatenate all label and prediction batches
    all_labels = np.concatenate(all_labels, axis=0)
    all_predictions = np.concatenate(all_predictions, axis=0)
    
    # Return metrics and the raw labels and predictions
    return all_labels, all_predictions


In [99]:
test_metrics = evaluate(test_dataloader, policy_net, classification_loss_fn)
test_true_labels, test_predictions = test_metrics

In [100]:
true_labels

array([[1., 0., 0., 0., 0.],
       [1., 1., 0., 0., 0.],
       [1., 0., 0., 0., 1.],
       ...,
       [0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0.]], dtype=float32)

In [101]:
predictions

array([[1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.],
       [1., 0., 1., 0., 0.],
       ...,
       [0., 0., 0., 0., 0.],
       [1., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0.]], dtype=float32)

In [102]:
# Flatten the arrays for metric calculation
true_labels1 = true_labels.copy()
predictions1 = predictions.copy()
true_labels_flat = true_labels.flatten()
predictions_flat = predictions.flatten()

In [103]:
# Macro metrics
accuracy_macro = accuracy_score(true_labels_flat, predictions_flat)
precision_macro = precision_score(true_labels_flat, predictions_flat, average='macro', zero_division=1)
recall_macro = recall_score(true_labels_flat, predictions_flat, average='macro', zero_division=1)
f1_macro = f1_score(true_labels_flat, predictions_flat, average='macro', zero_division=1)

# Micro metrics
accuracy_micro = accuracy_score(true_labels_flat, predictions_flat)
precision_micro = precision_score(true_labels_flat, predictions_flat, average='micro', zero_division=1)
recall_micro = recall_score(true_labels_flat, predictions_flat, average='micro', zero_division=1)
f1_micro = f1_score(true_labels_flat, predictions_flat, average='micro', zero_division=1)

# Weighted metrics
accuracy_weighted = accuracy_score(true_labels_flat, predictions_flat)
precision_weighted = precision_score(true_labels_flat, predictions_flat, average='weighted', zero_division=1)
recall_weighted = recall_score(true_labels_flat, predictions_flat, average='weighted', zero_division=1)
f1_weighted = f1_score(true_labels_flat, predictions_flat, average='weighted', zero_division=1)

# Print macro, micro, and weighted metrics
print("Macro Metrics:")
print(f"Accuracy: {accuracy_macro}, Precision: {precision_macro}, Recall: {recall_macro}, F1 Score: {f1_macro}")

print("\nMicro Metrics:")
print(f"Accuracy: {accuracy_micro}, Precision: {precision_micro}, Recall: {recall_micro}, F1 Score: {f1_micro}")

print("\nWeighted Metrics:")
print(f"Accuracy: {accuracy_weighted}, Precision: {precision_weighted}, Recall: {recall_weighted}, F1 Score: {f1_weighted}")

# Calculate and display confusion matrices for each label
label_names = [f"Label_{i+1}" for i in range(5)]  # Replace with actual label names if available
confusion_matrices = []

for i in range(5):  # Iterate over each label
    pred_i = predictions1[:, i]
    target_i = true_labels1[:, i]
    cm = confusion_matrix(target_i, pred_i)
    confusion_matrices.append(cm)
    
    # Print the confusion matrix
    print(f"\nConfusion Matrix for {label_names[i]}:\n{cm}\n")

Macro Metrics:
Accuracy: 0.8645112209965766, Precision: 0.8079539210575575, Recall: 0.7325075014447147, F1 Score: 0.7601850502639642

Micro Metrics:
Accuracy: 0.8645112209965766, Precision: 0.8645112209965766, Recall: 0.8645112209965766, F1 Score: 0.8645112209965766

Weighted Metrics:
Accuracy: 0.8645112209965766, Precision: 0.8551637971784889, Recall: 0.8645112209965766, F1 Score: 0.8551134425257381

Confusion Matrix for Label_1:
[[1120  364]
 [ 418  727]]


Confusion Matrix for Label_2:
[[1862   36]
 [ 368  363]]


Confusion Matrix for Label_3:
[[2257   46]
 [ 181  145]]


Confusion Matrix for Label_4:
[[2377   22]
 [ 179   51]]


Confusion Matrix for Label_5:
[[2401   32]
 [ 135   61]]



In [104]:
# Flatten the arrays for metric calculation
true_labels2 = test_true_labels.copy()
predictions2 = test_predictions.copy()
true_labels_flat = test_true_labels.flatten()
predictions_flat = test_predictions.flatten()

In [105]:
# Macro metrics
accuracy_macro = accuracy_score(true_labels_flat, predictions_flat)
precision_macro = precision_score(true_labels_flat, predictions_flat, average='macro', zero_division=1)
recall_macro = recall_score(true_labels_flat, predictions_flat, average='macro', zero_division=1)
f1_macro = f1_score(true_labels_flat, predictions_flat, average='macro', zero_division=1)

# Micro metrics
accuracy_micro = accuracy_score(true_labels_flat, predictions_flat)
precision_micro = precision_score(true_labels_flat, predictions_flat, average='micro', zero_division=1)
recall_micro = recall_score(true_labels_flat, predictions_flat, average='micro', zero_division=1)
f1_micro = f1_score(true_labels_flat, predictions_flat, average='micro', zero_division=1)

# Weighted metrics
accuracy_weighted = accuracy_score(true_labels_flat, predictions_flat)
precision_weighted = precision_score(true_labels_flat, predictions_flat, average='weighted', zero_division=1)
recall_weighted = recall_score(true_labels_flat, predictions_flat, average='weighted', zero_division=1)
f1_weighted = f1_score(true_labels_flat, predictions_flat, average='weighted', zero_division=1)

# Print macro, micro, and weighted metrics
print("Macro Metrics:")
print(f"Accuracy: {accuracy_macro}, Precision: {precision_macro}, Recall: {recall_macro}, F1 Score: {f1_macro}")

print("\nMicro Metrics:")
print(f"Accuracy: {accuracy_micro}, Precision: {precision_micro}, Recall: {recall_micro}, F1 Score: {f1_micro}")

print("\nWeighted Metrics:")
print(f"Accuracy: {accuracy_weighted}, Precision: {precision_weighted}, Recall: {recall_weighted}, F1 Score: {f1_weighted}")

# Calculate and display confusion matrices for each label
label_names = [f"Label_{i+1}" for i in range(5)]  # Replace with actual label names if available
confusion_matrices = []

for i in range(5):  # Iterate over each label
    pred_i = predictions1[:, i]
    target_i = true_labels1[:, i]
    cm = confusion_matrix(target_i, pred_i)
    confusion_matrices.append(cm)
    
    # Print the confusion matrix
    print(f"\nConfusion Matrix for {label_names[i]}:\n{cm}\n")

Macro Metrics:
Accuracy: 0.8631418790414607, Precision: 0.8009300948089731, Recall: 0.7377885612176214, F1 Score: 0.7621291179876081

Micro Metrics:
Accuracy: 0.8631418790414607, Precision: 0.8631418790414607, Recall: 0.8631418790414607, F1 Score: 0.8631418790414607

Weighted Metrics:
Accuracy: 0.8631418790414607, Precision: 0.8540285223552345, Recall: 0.8631418790414607, F1 Score: 0.8551584983231023

Confusion Matrix for Label_1:
[[1120  364]
 [ 418  727]]


Confusion Matrix for Label_2:
[[1862   36]
 [ 368  363]]


Confusion Matrix for Label_3:
[[2257   46]
 [ 181  145]]


Confusion Matrix for Label_4:
[[2377   22]
 [ 179   51]]


Confusion Matrix for Label_5:
[[2401   32]
 [ 135   61]]

